# 03 · Shared vs persona-specific variance

For each trait we split its per-persona vectors into a **shared** direction (common
to all personas) and **persona-specific residuals**. The *shared variance ratio* is
the fraction of total signal explained by the shared direction — lower means more
context-dependent. This reproduces the headline table in
[`docs/overview.md`](../docs/overview.md).

In [ ]:
import torch
from pathlib import Path
from persona_steering.utils import parse_persona_trait_from_stem

def load_vector_dir(vectors_dir, layer=22):
    """Load all steering vectors from a dir of .pt files at one layer.

    Robust to two on-disk formats:
      * dict payload  {'vector': Tensor[L, D], 'persona':..., 'trait':...}
      * a raw Tensor[L, D]
    Returns {(persona, trait): Tensor[D]} for the requested layer.
    """
    vectors_dir = Path(vectors_dir)
    out = {}
    for pt in sorted(vectors_dir.glob('*.pt')):
        obj = torch.load(pt, map_location='cpu', weights_only=False)
        vec = obj['vector'] if isinstance(obj, dict) and 'vector' in obj else obj
        vec = vec.float()
        persona, trait = parse_persona_trait_from_stem(pt.stem)
        if persona and trait and layer < vec.shape[0]:
            out[(persona, trait)] = vec[layer]
    return out

In [ ]:
from pathlib import Path
from persona_steering.config import Trait
from persona_steering.utils import VectorShim
from persona_steering.analysis import decompose_shared_specific

DATA, LAYER = Path('hf_data'), 22

def shared_variance_table(vectors_dir):
    flat = load_vector_dir(vectors_dir, layer=LAYER)
    traits = sorted({t for (_, t) in flat})
    table = {}
    for t in traits:
        per_persona = {p: VectorShim(v, p, t, LAYER)
                       for (p, tt), v in flat.items() if tt == t}
        dec = decompose_shared_specific(per_persona)
        table[t] = dec.variance_explained
    return table

iv_tab = shared_variance_table(DATA / 'vectors')
caa_tab = shared_variance_table(DATA / 'caa_vectors')

In [ ]:
print(f"{'trait':14s} {'IV':>8s} {'CAA':>8s}")
for t in sorted(iv_tab, key=lambda k: -iv_tab[k]):
    print(f'{t:14s} {iv_tab[t]*100:7.1f}% {caa_tab.get(t, float("nan"))*100:7.1f}%')

**Reading it:** under IV, RLHF-shaped traits (honesty, assertiveness, confidence…)
sit above ~80% shared variance; impulsivity and risk-taking sit below. CAA vectors
are consistently *less* universal — they carry more persona-entangled signal.

➡️ Next: **[04 · Reproduce figures](04_reproduce_figures.ipynb)**.